In [8]:
import pandas as pd
import pdal
import json
import geopandas as gpd
import numpy as np
import glob
import sys
import os
import shutil
# from multiprocessing import Pool
from multiprocessing.dummy import Pool
import numpy as np

In [9]:
gdf_articulacao = gpd.read_file('results/folhas_sp_cortada.gpkg')

In [10]:
gdf_articulacao.to_crs(epsg=31983, inplace=True)

In [11]:
DATA_DIR_2024 = '/Users/fernandogomes/dev/LiDAR-Sampa-2024'
RESULT_FOLDER = '/Users/fernandogomes/dev/LiDAR_produtos'

In [12]:
class Scm:

    def __init__(self, scm) -> None:
        
        coords = [[xy[0], xy[1]] for xy in gdf_articulacao.set_index("nome").loc[scm].geometry.exterior.coords]
        xy_max = np.max(np.array(coords), axis=0) 
        xy_min = np.min(np.array(coords), axis=0)
        width, height = np.ceil(xy_max * 2) - np.ceil(xy_min * 2)

        origin_x, origin_y = np.floor(xy_min * 2)/2
        
        self.scm = scm
        self.width = width
        self.height = height
        self.origin_x = origin_x
        self.origin_y = origin_y

In [13]:
def pipeline(scm, ano):
    # Retorna o json com o Pipeline para o determinado SCM
    scm_att = Scm(scm)
    pipeline = [
        {
            "type": "readers.las",
            "filename": f'temp/{ano}-{scm}.laz',
            "override_srs": "EPSG:31983"
        },
        {
            "type": "filters.outlier",
            "method": "statistical",
            "mean_k": 8,
            "multiplier": 2.0
        }, 
        {
            "filename":f"{RESULT_FOLDER}/{ano}/MDS_sem_vegetacao/MDS-sem-vegetacao-{scm}-{ano}.tiff",
            "gdaldriver":"GTiff",
            "output_type":"mean",
            "resolution":"1",
            "type": "writers.gdal",
            "gdalopts":"COMPRESS=ZSTD, PREDICTOR=3, BIGTIFF=YES",
            "width": scm_att.width,
            "height": scm_att.height,
            "origin_x": scm_att.origin_x,
            "origin_y": scm_att.origin_y,
            "nodata":"0",
            "data_type": "float32",
            "where": "(Classification != 3 && Classification != 4 && Classification != 5)",
            "default_srs": "EPSG:31983"
        },
    ]
    return pipeline

In [14]:
def processo(scm):

    if len(glob.glob(f"{RESULT_FOLDER}/2024/MDS_sem_vegetacao/MDS-sem-vegetacao-{scm}-2024.tiff")) > 0:
        # print(f'SCM {scm} processado anteriormente')
        return None

    # Copia arquivos de 2017 e 2020 para uma pasta temporária
    file_2024 = glob.glob(f'{DATA_DIR_2024}/*{scm}*.laz')
    if len(file_2024) != 1:
        raise ValueError(f'Os arquivos do {scm} parecem não conforme!')
    
    print(f'Processando {scm}')

    shutil.copy(file_2024[0], f'temp/2024-{scm}.laz')
  
    # Processa o PDAL para cada ano: MDT, MDS, BHM, VHM
    mdt_mds = pdal.Pipeline(json.dumps(pipeline(scm, 2024)))
    n_points = mdt_mds.execute()
    # print(f'Executando MDT/MDS com {n_points} pontos')

    # Exclui os arquivos da pasta temporária
    os.remove(f'temp/2024-{scm}.laz')
    
    print(f'Processado {scm}')
    
    return None

In [15]:
def processa_tudo():
    # Itera sobre todos os SCMs
    # Utilizando multiprocessamento
    scms = gdf_articulacao.loc[:, 'nome'].to_list()
    with Pool(12) as p:
        _ = p.starmap(processo, zip(scms))
    # for scm in scms:
    #     processo(scm)
    return None

In [16]:
processa_tudo()

Processando Y-C-VI-3-SE-D-I-3
Processando Y-C-VI-1-NE-F-II-6
Processando Y-C-VI-4-SO-A-I-5
Processando Y-C-VI-2-NO-E-I-1
Processando Y-C-VI-1-NE-F-II-2
Processando Y-C-VI-3-SE-B-III-2
Processando Y-C-VI-4-SO-A-III-1
Processando Y-C-VI-1-SE-D-IV-1
Processando Y-C-VI-3-SE-B-IV-6
Processando Y-C-III-3-SE-C-II-2
Processando Y-C-III-3-SE-D-I-6
Processando Y-C-VI-3-SE-B-IV-5
Processado Y-C-VI-1-SE-D-IV-1
Processando Y-C-VI-1-SE-D-II-4
Processado Y-C-III-3-SE-D-I-6
Processando Y-C-III-3-SE-D-I-3
Processado Y-C-VI-1-NE-F-II-2
Processando Y-C-VI-1-NE-D-IV-5
Processado Y-C-VI-1-NE-F-II-6
Processando Y-C-VI-1-NE-F-II-3
Processado Y-C-VI-3-SE-B-IV-6
Processando Y-C-VI-3-SE-B-IV-3
Processado Y-C-VI-2-NO-E-I-1
Processando Y-C-VI-2-NO-C-III-4
Processado Y-C-VI-3-SE-B-III-2
Processando Y-C-VI-3-SE-B-I-5
Processado Y-C-VI-3-SE-B-IV-5
Processando Y-C-VI-3-SE-B-IV-2
Processado Y-C-III-3-SE-C-II-2
Processando Y-C-III-3-SE-A-IV-5
Processado Y-C-VI-4-SO-A-III-1
Processando Y-C-VI-4-SO-A-I-4
Processado Y-C-V